# P7 – Member D: LLM judge + new-generator test set
Run the cells top to bottom. **Every API-calling step is resumable**: after a crash or an empty
balance, just rerun the same cell and only the missing rows are processed.

> Put the `.py` files in the same directory as this notebook.
>
> Judge model: `deepseek-flash`  
> Generator 1: `deepseek-chat`  
> Generator 2: `deepseek-reasoner`

In [ ]:
# 0. Install dependencies + set the API key (never hard-code or commit the key)
# !pip install -r requirements/P7_D.txt
import os, getpass
os.environ['DEEPSEEK_API_KEY'] = os.environ.get('DEEPSEEK_API_KEY') or getpass.getpass('DeepSeek key: ')
os.environ['P7_D_MODEL'] = 'deepseek-flash'        # judge model (must differ from both generators)
os.environ['P7_GEN_MODEL_1'] = 'deepseek-chat'     # new generator 1
os.environ['P7_GEN_MODEL_2'] = 'deepseek-reasoner' # new generator 2
os.makedirs('data/hc3_raw', exist_ok=True); os.makedirs('results', exist_ok=True); os.makedirs('requirements', exist_ok=True)

## 1. Data: five English HC3 domains -> normalisation -> fixed split (val / test / few-shot / new-generator prompts)
First put `reddit_eli5.jsonl, open_qa.jsonl, wiki_csai.jsonl, medicine.jsonl, finance.jsonl` into `data/hc3_raw/` (from https://huggingface.co/datasets/Hello-SimpleAI/HC3).

In [ ]:
# Check that the raw HC3 files are present before running P7_data.py
from pathlib import Path
need = ['reddit_eli5.jsonl', 'open_qa.jsonl', 'wiki_csai.jsonl', 'medicine.jsonl', 'finance.jsonl']
missing = [f for f in need if not (Path('data/hc3_raw') / f).exists()]
if missing:
    raise SystemExit(f'Missing raw HC3 files in data/hc3_raw/: {missing}\n'
                     'Download them from https://huggingface.co/datasets/Hello-SimpleAI/HC3/tree/main')
print('raw HC3 files OK')

In [ ]:
!python P7_data.py

In [ ]:
# Consistency check: normalize_text in predict must be identical to the one in P7_data
import P7_data, P7_D_predict
for s in ['A [1] “q” — x\n\n\n\n- b  c.', 'Plain text.']:
    assert P7_data.normalize_text(s) == P7_D_predict.normalize_text(s)
print('normalize OK')

## 2. Shortcut audit (show/exclude that the score comes from formatting artifacts)
If `RAW` is high and `NORMALISED` drops clearly, normalisation works. Put both numbers on the slide.

In [ ]:
!python P7_D_diagnostics.py --input data/P7_hc3_test.csv

## 3. Compare the 3 prompts on the VALIDATION set (prompt selection uses validation only)
Run with `--limit 20` first to confirm the key/model works, then run the full set.

In [ ]:
!python P7_D_experiment.py --input data/P7_hc3_val.csv --tag val --limit 20

In [ ]:
!python P7_D_experiment.py --input data/P7_hc3_val.csv --tag val

In [ ]:
!python P7_D_report.py
print('Best prompt =', open('results/P7_D_best_mode.txt').read())

## 4. Test set: best prompt only, run once

In [ ]:
BEST = open('results/P7_D_best_mode.txt').read().strip()
print('Using BEST =', BEST)
!python P7_D_experiment.py --input data/P7_hc3_test.csv --tag test --modes {BEST} --val_tag val

## 5. New-generator test set (send `data/P7_new_generators.csv` to A, B, C, E before Day 4)

In [ ]:
!python P7_D_generate.py

In [ ]:
!python P7_D_diagnostics.py --input data/P7_new_generators.csv

In [ ]:
# Length-confound check: single-feature AUROC of n_words on the new-generator set
import pandas as pd
audit = pd.read_csv('results/P7_D_shortcut_audit.csv')
print(audit[audit.file.str.contains('new_generators')][['ai_source', 'text_version', 'format_only_AUROC', 'strongest_single_feature']])

In [ ]:
!python P7_D_experiment.py --input data/P7_new_generators.csv --tag newgen --modes {BEST} --val_tag val

## 6. Summary table / figures / reason examples / cost

In [ ]:
# Replace 0.0 with the current DeepSeek price in USD per 1M tokens (input / output)
PRICE_IN  = 0.0
PRICE_OUT = 0.0
!python P7_D_report.py --price_in {PRICE_IN} --price_out {PRICE_OUT}
import pandas as pd
display(pd.read_csv('results/P7_D_main_table.csv'))
print(open('results/P7_D_cost.txt').read())

In [ ]:
from IPython.display import Image, display
display(Image('results/P7_D_fig_prompts.png')); display(Image('results/P7_D_fig_scores.png'))
display(pd.read_csv('results/P7_D_reason_examples.csv'))

## 7. Demo texts (hand to F) and agreement analysis (wait for A/B/C predictions)

In [ ]:
# Write the demo texts into results/ so that they sit next to the meta file
# Add --attack_csv results/attack_pairs.csv if E's attack pairs are available
!python P7_D_demo.py --out results/P7_demo_texts.csv

In [ ]:
# Run after A/B/C each provide a uid,prob_ai csv:
# !python P7_D_agreement.py --truth results/P7_D_pred_test.csv --others A=results/P7_A_test.csv B=results/P7_B_test.csv C=results/P7_C_test.csv --tag test --data data/P7_hc3_test.csv

In [ ]:
# 8. Dependency versions (written to requirements/P7_D.txt)
!python P7_D_freeze.py > requirements/P7_D.txt
print(open('requirements/P7_D.txt').read())